<a href="https://colab.research.google.com/github/Midunvarsan/Codtech-Intern/blob/main/Generative_Text_Model_Explanation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
# Task 4: Generative Text Model for CODTECH IT SOLUTIONS Internship Portfolio

This Jupyter Notebook presents a generative text model built using Hugging Face's `transformers` library, specifically leveraging the GPT-2 architecture. This project fulfills 'TASK 4: GENERATIVE TEXT MODEL' for the CODTECH IT SOLUTIONS artificial intelligence internship portfolio, demonstrating the ability to generate coherent paragraphs on specific topics from a user-provided prompt.

## 1. Environment Setup

This section handles the installation of necessary libraries and basic setup for the notebook.
"""

# Install necessary libraries
!pip install torch transformers ipywidgets gradio sentencepiece -q

# Import necessary libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import ipywidgets as widgets
from IPython.display import display, HTML
import gradio as gr

"""
## 2. Model and Tokenizer Initialization

Here, we initialize the pre-trained GPT-2 model and its corresponding tokenizer from Hugging Face. We'll use 'gpt2-medium' for a balance of performance and resource usage.
"""

# Define the model name
MODEL_NAME = "gpt2-medium"

# Load pre-trained tokenizer
# The tokenizer is responsible for converting text into numerical tokens that the model can understand.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load pre-trained model
# AutoModelForCausalLM is used for models that predict the next token in a sequence,
# which is the core functionality for generative text models.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set the pad_token_id for the tokenizer.
# Some models might not have a default pad_token. If not set, it can cause issues during batch processing
# or when padding is required for generation. For GPT-2, we often use the end-of-sequence token as padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print(f"Model '{MODEL_NAME}' and Tokenizer initialized successfully.")

"""
## 3. Text Generation Function

This section defines a function that encapsulates the text generation logic. It includes configurable parameters to control the quality and creativity of the generated text.
"""
def generate_text(prompt: str,
                  max_length: int = 150,
                  temperature: float = 0.7,
                  top_k: int = 50,
                  top_p: float = 0.95,
                  repetition_penalty: float = 1.2,
                  num_return_sequences: int = 1,
                  seed: int = 42) -> list:
    """
    Generates text based on a given prompt using the initialized GPT-2 model.

    Args:
        prompt (str): The initial text prompt to start generation.
        max_length (int): The maximum number of tokens to generate, including the prompt.
                          Controls the length of the generated output.
        temperature (float): Controls the randomness of predictions.
                             Higher values (e.g., 1.0) make the output more random/creative.
                             Lower values (e.g., 0.5) make it more deterministic/focused.
                             A good range is typically between 0.5 and 1.0.
        top_k (int): In top-k sampling, the model only considers the top 'k' most likely next tokens.
                     This helps to reduce the risk of generating incoherent text by filtering out
                     very unlikely tokens.
        top_p (float): In nucleus (top-p) sampling, the model considers the smallest set of tokens
                       whose cumulative probability exceeds 'p'. This provides a dynamic vocabulary
                       size, allowing for more diverse generation than top-k while still avoiding
                       low-probability tokens.
        repetition_penalty (float): Penalizes tokens that have already appeared in the text.
                                    A value greater than 1.0 reduces the likelihood of repeating
                                    phrases or words, preventing repetitive loops in the generated text.
        num_return_sequences (int): The number of independent sequences to generate.
        seed (int): A random seed for reproducibility.

    Returns:
        list: A list of generated text strings.
    """
    # Set a random seed for reproducibility
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Encode the prompt text into numerical tokens that the model can process.
    # return_tensors='pt' ensures the output is a PyTorch tensor.
    input_ids = tokenizer.encode(prompt, return_tensors='pt')

    # Generate text using the model.
    # The `generate` method handles the full generation process.
    # `attention_mask`: Used to ignore padding tokens during attention calculations.
    #                   It tells the model which tokens are actual content and which are just padding.
    #                   For a single prompt, it's a tensor of ones.
    output = model.generate(
        input_ids,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id, # Ensure padding token is explicitly set for generation
        do_sample=True, # Enable sampling to use temperature, top_k, top_p
        early_stopping=True # Stop generation once all samples are finished
    )

    # Decode the generated tokens back into human-readable text.
    # skip_special_tokens=True prevents decoding special tokens like [CLS], [SEP], [PAD].
    generated_texts = [tokenizer.decode(output_id, skip_special_tokens=True) for output_id in output]

    return generated_texts

"""
## 4. Demonstrating Execution

This section shows a basic demonstration of the text generation function with a predefined prompt and prints the output.
"""

# Define a seed prompt
seed_prompt = "The future of artificial intelligence in healthcare is expected to bring about transformative changes, particularly in areas such as"

# Generate text using the defined function and parameters
print(f"Generating text for prompt: \"{seed_prompt}\"\n")
generated_outputs = generate_text(
    prompt=seed_prompt,
    max_length=200,
    temperature=0.8,
    top_k=50,
    top_p=0.92,
    repetition_penalty=1.2
)

# Print the generated output cleanly
for i, text in enumerate(generated_outputs):
    print(f"--- Generated Text {i+1} ---")
    print(text)
    print("\n" + "="*50 + "\n")

"""
## 5. Interactive Testing with Gradio

This section provides an interactive interface using `Gradio`, allowing users to input custom prompts and adjust generation parameters on the fly.
"""

# Define a wrapper function for Gradio interface
def gradio_generate_text(prompt, max_length, temperature, top_k, top_p, repetition_penalty):
    # Call the core generation function
    generated_texts = generate_text(
        prompt=prompt,
        max_length=int(max_length),
        temperature=float(temperature),
        top_k=int(top_k),
        top_p=float(top_p),
        repetition_penalty=float(repetition_penalty)
    )
    # Gradio expects a single string or list of strings for output
    return "\n\n---\n\n".join(generated_texts)

# Create Gradio interface
iface = gr.Interface(
    fn=gradio_generate_text,
    inputs=[
        gr.Textbox(lines=3, label="Enter your prompt:", value="The emergence of quantum computing will revolutionize artificial intelligence by"),
        gr.Slider(minimum=50, maximum=500, value=150, label="Max Length (tokens)"),
        gr.Slider(minimum=0.1, maximum=2.0, value=0.7, label="Temperature (creativity)", step=0.05),
        gr.Slider(minimum=1, maximum=100, value=50, label="Top-K (diverse words)", step=1),
        gr.Slider(minimum=0.01, maximum=1.0, value=0.95, label="Top-P (nucleus sampling)", step=0.01),
        gr.Slider(minimum=1.0, maximum=2.0, value=1.2, label="Repetition Penalty", step=0.05)
    ],
    outputs="text",
    title="Generative Text Model (GPT-2 Medium)",
    description="Interact with the GPT-2 medium model to generate text. Adjust parameters to control creativity and coherence."
)

# Launch the Gradio interface
# To run this in Colab, you might need to click the public URL provided by Gradio
# iface.launch(share=True) # Use share=True to get a public link
iface.launch()

"""
## Repository Assets

Below are the contents for `requirements.txt` file, which would accompany this notebook in a GitHub repository.

### `requirements.txt`
"""
"""
torch>=1.10.0
transformers>=4.18.0
ipywidgets>=7.0.0
gradio>=3.0.0
sentencepiece>=0.1.91
"""


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model 'gpt2-medium' and Tokenizer initialized successfully.
Generating text for prompt: "The future of artificial intelligence in healthcare is expected to bring about transformative changes, particularly in areas such as"

--- Generated Text 1 ---
The future of artificial intelligence in healthcare is expected to bring about transformative changes, particularly in areas such as precision medicine and patient safety.
 (Source: Getty) Artificial Intelligence will change how we treat people with mental illnesses like depression, anxiety or post-traumatic stress disorder. There are three main ways AI can improve health care for patients – improved understanding of the body's physiology; better treatment decision making under real time conditions where treatments need only be applied once a day rather than over many hours during prolonged periods of monitoring/monitoring by staff using machine learning techniques used commonly on large scale medical imaging applications today; faster diagn

'\ntorch>=1.10.0\ntransformers>=4.18.0\nipywidgets>=7.0.0\ngradio>=3.0.0\nsentencepiece>=0.1.91\n'